# Full HIGGS Benchmark on Colab or Kaggle — Fixed (v5, smaller cells)

**Project:** Higgs Event Lab | **Repository:** [CosmicMagnetar/higgs-event-lab](https://github.com/CosmicMagnetar/higgs-event-lab)

Run cells from top to bottom. This version separates the longer setup, download, and training steps into smaller pieces. The UCI HIGGS archive is large (about 2.6 GB). If a download fails, rerun the download cell; partial data is retained and resumed when the server supports it.

**Tip:** After a failed cell, do not restart the runtime unless necessary. Re-run that cell, then continue in order.

## Step 1A — Imports and project settings

Run once at the beginning. This cell only sets the repository path and URL. It also records the runtime assumptions needed for later cells and keeps the workflow reproducible on both Colab and Kaggle.


In [3]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/CosmicMagnetar/higgs-event-lab.git"
root = Path(os.environ.get("HIGGS_REPO_ROOT", Path.cwd() / "higgs-event-lab")).resolve()
print("Repository path:", root)

Repository path: /kaggle/working/higgs-event-lab


## Step 1B — Clone the repository if needed

If the source folder is already present, this skips cloning. The cell is safe to rerun after an interrupted session and prevents accidental use of an incomplete directory.


In [4]:
if not (root / "src" / "higgs_event_lab").exists():
    root.parent.mkdir(parents=True, exist_ok=True)
    if root.exists() and any(root.iterdir()):
        raise RuntimeError(
            f"The repository path exists but does not contain src/higgs_event_lab: {root}. "
            "Choose a clean repository path or remove the incomplete folder, then rerun this cell."
        )
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)

os.environ["HIGGS_REPO_ROOT"] = str(root)
os.chdir(root)
print("Repository ready:", root)

Repository ready: /kaggle/working/higgs-event-lab


## Step 1C — Install project requirements

Run this after Step 1B. Package installation can take a few minutes. Installing the pinned project dependency set ensures that the repository modules and notebook commands use the same environment.


In [5]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(root / "requirements.txt")], check=True)
src_dir = root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
print("Dependencies installed; project source path added.")

Dependencies installed; project source path added.


## Step 2A — Set up the HIGGS download paths

This cell prepares paths and imports the streaming download library. It does not download the dataset yet.

In [6]:
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

root = Path(os.environ.get("HIGGS_REPO_ROOT", Path.cwd() / "higgs-event-lab")).resolve()
raw_path = root / "data" / "raw" / "HIGGS.csv.gz"
raw_path.parent.mkdir(parents=True, exist_ok=True)
url = "https://archive.ics.uci.edu/static/public/280/higgs.zip"
zip_path = raw_path.parent / "higgs_download.zip"
temp_raw_path = raw_path.with_name(raw_path.name + ".part")

try:
    import requests
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"], check=True)
    import requests

print("Download destination:", zip_path)
print("Extracted data destination:", raw_path)

Download destination: /kaggle/working/higgs-event-lab/data/raw/higgs_download.zip
Extracted data destination: /kaggle/working/higgs-event-lab/data/raw/HIGGS.csv.gz


## Step 2B — Add a ZIP validation helper

This small helper checks whether the downloaded ZIP has a readable directory and contains a `.csv.gz` file.

In [7]:
def find_csv_gz_member(archive_path: Path):
    """Return the CSV.GZ member name if the ZIP directory is readable."""
    try:
        with zipfile.ZipFile(archive_path, "r") as archive:
            return next(
                (name for name in archive.namelist() if name.lower().endswith(".csv.gz")),
                None,
            )
    except (OSError, zipfile.BadZipFile, EOFError):
        return None

print("ZIP validation helper is ready.")

ZIP validation helper is ready.


## Step 2C — Add retry and resume download logic

This defines the downloader only; the next cell actually starts the download. Interrupted bytes are kept, and HTTP Range resume is used when supported by the UCI server.

In [8]:
def download_zip_resumable(url, destination, max_attempts=10):
    session = requests.Session()
    last_error = None
    for attempt in range(1, max_attempts + 1):
        if find_csv_gz_member(destination):
            print("A readable ZIP archive is already present; reusing it.")
            return
        existing = destination.stat().st_size if destination.exists() else 0
        headers = {"Range": f"bytes={existing}-"} if existing else {}
        try:
            with session.get(url, headers=headers, stream=True, timeout=(30, 180)) as response:
                if response.status_code == 416:
                    if existing and find_csv_gz_member(destination):
                        return
                    destination.unlink(missing_ok=True)
                    raise IOError("Server rejected the resume offset")
                response.raise_for_status()
                mode = "ab" if response.status_code == 206 and existing else "wb"
                expected = response.headers.get("Content-Length")
                received = 0
                with open(destination, mode) as target:
                    for chunk in response.iter_content(chunk_size=4 * 1024 * 1024):
                        if chunk:
                            target.write(chunk)
                            received += len(chunk)
                if expected is not None and received != int(expected):
                    raise IOError(f"Incomplete response: {received} of {expected} bytes")
            if find_csv_gz_member(destination):
                print(f"ZIP is readable ({destination.stat().st_size / (1024**3):.2f} GiB).")
                return
            raise IOError("ZIP directory is not complete yet")
        except Exception as exc:
            last_error = exc
            print(f"Attempt {attempt}/{max_attempts} failed: {exc}")
            if attempt < max_attempts:
                time.sleep(min(2 ** (attempt - 1), 30))
    raise RuntimeError(f"Could not download a readable ZIP. Rerun this cell. Last error: {last_error}")

print("Retry/resume downloader is ready.")

Retry/resume downloader is ready.


## Step 2D — Download the ZIP archive

Run this cell to start or resume the download. It skips downloading if the extracted CSV already exists or a valid ZIP is already saved.

In [9]:
if raw_path.exists():
    print("Raw HIGGS data already exists; skipping download.")
elif find_csv_gz_member(zip_path):
    print("A readable ZIP archive already exists; ready to extract.")
else:
    print("Downloading the full HIGGS archive from UCI. This is a large file.")
    download_zip_resumable(url, zip_path, max_attempts=10)

if raw_path.exists():
    print("Raw file:", raw_path, f"({raw_path.stat().st_size / (1024**3):.2f} GiB)")
else:
    print("ZIP ready:", zip_path if find_csv_gz_member(zip_path) else "download incomplete; rerun this cell")

Attempt 1/10; partial file: 1,256.4 MiB
ZIP is readable (2.62 GiB).
ZIP ready: /kaggle/working/higgs-event-lab/data/raw/higgs_download.zip


## Step 2E — Extract and validate the dataset

This cell extracts to a temporary file and publishes it only after extraction completes. If extraction fails, the corrupt ZIP is deleted so a fresh download can be tried.

In [10]:
if raw_path.exists():
    print("Raw HIGGS data already exists; skipping extraction.")
else:
    gz_name = find_csv_gz_member(zip_path)
    if not gz_name:
        raise RuntimeError("No valid ZIP archive found. Run Step 2D again to download or resume it.")

    print(f"Extracting {gz_name} ...")
    temp_raw_path.unlink(missing_ok=True)
    try:
        with zipfile.ZipFile(zip_path, "r") as archive:
            with archive.open(gz_name, "r") as source, open(temp_raw_path, "wb") as target:
                shutil.copyfileobj(source, target, length=4 * 1024 * 1024)
        if not temp_raw_path.is_file() or temp_raw_path.stat().st_size == 0:
            raise IOError("Extracted CSV.GZ is empty.")
        with open(temp_raw_path, "rb") as extracted:
            if extracted.read(2) != b"\x1f\x8b":
                raise IOError("Extracted file does not have a valid gzip header.")
        temp_raw_path.replace(raw_path)
    except Exception as exc:
        temp_raw_path.unlink(missing_ok=True)
        zip_path.unlink(missing_ok=True)
        raise RuntimeError(
            "Extraction validation failed. The broken ZIP was removed; rerun Step 2D, "
            "and check disk space and network stability."
        ) from exc
    else:
        zip_path.unlink(missing_ok=True)

print("Raw data ready:", raw_path)
print(f"Raw file size: {raw_path.stat().st_size / (1024**3):.2f} GiB")

Extracting HIGGS.csv.gz ...
Raw data ready: /kaggle/working/higgs-event-lab/data/raw/HIGGS.csv.gz
Raw file size: 2.62 GiB


## Step 3A — Set up the processed sample paths

These files are derived from the raw dataset. Run this after Step 2E. Keeping both paths explicit makes it clear which sample supports the full benchmark and which sample supports EDA.


In [11]:
import os
import sys
from pathlib import Path

root = Path(os.environ.get("HIGGS_REPO_ROOT", Path.cwd() / "higgs-event-lab")).resolve()
src_dir = root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

raw_path = root / "data" / "raw" / "HIGGS.csv.gz"
out_path = root / "data" / "processed" / "higgs_sample_1m.csv.gz"
sample_path = root / "data" / "processed" / "higgs_sample_250k.csv.gz"
if not raw_path.is_file():
    raise FileNotFoundError(f"Raw dataset is missing: {raw_path}. Finish Step 2 first.")
print("One-million-row sample:", out_path)
print("250,000-row sample:", sample_path)

One-million-row sample: /kaggle/working/higgs-event-lab/data/processed/higgs_sample_1m.csv.gz
250,000-row sample: /kaggle/working/higgs-event-lab/data/processed/higgs_sample_250k.csv.gz


## Step 3B — Create the one-million-row training sample

This is the main sample used by the training step. Existing output is reused. The extraction uses the documented seed so the same rows can be audited and regenerated from the raw file.


In [12]:
from higgs_event_lab.data import extract_sample

if not out_path.exists():
    extract_sample(
        source_path=raw_path,
        output_path=out_path,
        n_rows=1_000_000,
        seed=42,
        compare_later_chunk=False,
    )
else:
    print("Training sample already exists; skipping extraction.")

print("Training sample ready:", out_path)

Training sample ready: /kaggle/working/higgs-event-lab/data/processed/higgs_sample_1m.csv.gz


## Step 3C — Create the smaller EDA sample

The 250,000-row sample is used for exploratory plots and audits. The smaller sample keeps plotting and validation practical while preserving the project split and sampling protocol.


In [13]:
if not sample_path.exists():
    extract_sample(
        source_path=raw_path,
        output_path=sample_path,
        n_rows=250_000,
        seed=42,
        compare_later_chunk=True,
    )
else:
    print("EDA sample already exists; skipping extraction.")

print("EDA sample ready:", sample_path)

EDA sample ready: /kaggle/working/higgs-event-lab/data/processed/higgs_sample_250k.csv.gz


## Step 3D — Generate saved EDA evidence

This runs the project's analysis script and writes plots and tables under `outputs/eda`.

### EDA fallback helper

The cloned public repository may not yet contain the optional report helper script. This local fallback creates the same essential report tables and figures directly in the notebook.


In [ ]:
def generate_eda_fallback(sample_path, eda_dir):
    import json
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import pandas as pd
    from higgs_event_lab.data import load_higgs_sample
    batch = load_higgs_sample(sample_path, n_rows=250000, feature_group="all")
    X, y = batch.X, batch.y.to_numpy()
    eda_dir.mkdir(parents=True, exist_ok=True)
    stats = X.describe().T
    stats["missing_count"] = X.isna().sum()
    stats["signal_mean"] = X.loc[y == 1].mean()
    stats["background_mean"] = X.loc[y == 0].mean()
    stats.to_csv(eda_dir / "feature_summary.csv")
    btag = {name: sorted(float(value) for value in X[name].unique()) for name in X.columns if "btag" in name}
    (eda_dir / "btag_summary.json").write_text(json.dumps(btag, indent=2), encoding="utf-8")
    corr = X.corr()
    corr.to_csv(eda_dir / "correlation_matrix.csv")
    means = pd.DataFrame({"signal_mean": X.loc[y == 1].mean(), "background_mean": X.loc[y == 0].mean()}).rename_axis("feature").reset_index()
    means.to_csv(eda_dir / "class_conditional_means.csv", index=False)
    pd.Series(y).value_counts().sort_index().rename_axis("class_label").reset_index(name="events").to_csv(eda_dir / "class_balance.csv", index=False)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(["Background (0)", "Signal (1)"], [(y == 0).sum(), (y == 1).sum()], color=["#5f6b7a", "#24558f"])
    ax.set_ylabel("Events")
    ax.set_title("Class balance in the EDA sample")
    fig.tight_layout()
    fig.savefig(eda_dir / "class_balance.png", dpi=160)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(9, 7))
    image = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
    ax.set_title("Pearson correlation matrix")
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(eda_dir / "correlation_heatmap.png", dpi=160)
    plt.close(fig)
    summary = {"rows": int(len(X)), "feature_count": int(X.shape[1]), "signal_fraction": float((y == 1).mean()), "missing_values": int(X.isna().sum().sum()), "figures": ["class_balance.png", "correlation_heatmap.png"]}
    (eda_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("EDA fallback completed:", eda_dir)

### Run the EDA step

Use the saved-script path when the cloned repository includes it; otherwise the fallback helper above runs the analysis directly and creates the report artifacts.


In [ ]:
eda_script = root / "scripts" / "run_eda.py"
eda_result = None
if eda_script.is_file():
    eda_result = subprocess.run([sys.executable, str(eda_script), "--data", str(sample_path), "--rows", "250000", "--output-dir", str(root / "outputs" / "eda")], cwd=root, check=False)
if eda_result is None or eda_result.returncode != 0:
    generate_eda_fallback(sample_path, root / "outputs" / "eda")
else:
    print("EDA exit code:", eda_result.returncode)


## Step 3E — Reconcile the split and saved predictions

This audit checks raw-file details, duplicates, shared test indices, and saved predictions. It may be run after training artifacts exist.

In [ ]:
audit_script = root / "scripts" / "reconcile_results.py"
audit_result = None
if audit_script.is_file():
    audit_result = subprocess.run([sys.executable, str(audit_script), "--sample", str(sample_path), "--raw", str(raw_path), "--output", str(root / "outputs" / "reconciliation_report.json")], cwd=root, check=False)
if audit_result is None or audit_result.returncode != 0:
    import gzip
    import hashlib
    import json
    import pandas as pd
    from higgs_event_lab.features import ALL_COLUMNS
    from higgs_event_lab.protocol import SplitConfig, create_stratified_splits
    frame = pd.read_csv(sample_path, header=None, names=list(ALL_COLUMNS))
    labels = frame["class_label"].to_numpy(dtype="int64")
    split = create_stratified_splits(labels, SplitConfig(seed=42))
    parts = {"train": split.train, "validation": split.validation, "test": split.test}
    hashes = pd.util.hash_pandas_object(frame.drop(columns=["class_label"]), index=False).to_numpy()
    hash_sets = {name: set(hashes[indexes].tolist()) for name, indexes in parts.items()}
    pairs = [("train", "validation"), ("train", "test"), ("validation", "test")]
    duplicate_overlap = sum(len(hash_sets[a] & hash_sets[b]) for a, b in pairs)
    prediction_paths = sorted((root / "outputs").glob("*_*/test_predictions.csv"))
    prediction_sets = [set(pd.read_csv(path)["row_index"].astype(int)) for path in prediction_paths]
    common_test_indices = len(set.intersection(*prediction_sets)) if prediction_sets else 0
    digest = hashlib.sha256()
    raw_rows = 0
    with gzip.open(raw_path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
            raw_rows += chunk.count(b"\n")
    report = {"sample": {"rows": int(len(frame)), "columns": int(frame.shape[1])}, "split": split.metadata(), "partition_audit": {"duplicate_feature_rows_across_partitions": int(duplicate_overlap), "partitions_disjoint": duplicate_overlap == 0}, "prediction_audit": {"files_checked": len(prediction_paths), "shared_test_index_count": int(common_test_indices)}, "raw_file": {"rows": raw_rows, "sha256": digest.hexdigest(), "size_bytes": raw_path.stat().st_size}}
    audit_path = root / "outputs" / "reconciliation_report.json"
    audit_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print("Reconciliation fallback completed:", audit_path)
else:
    print("Reconciliation exit code:", audit_result.returncode)


## Research and authorship

Saumya Mishra owns the physics background, dataset guide, and QA plan that explain why the 21 low-level and 7 high-level features are the right problem here. CosmicMagnetar tuned the models and prepared this Colab walkthrough so the numbers can be reproduced without claiming physics the benchmark never measures. Use this note in your report and viva so the reader understands who wrote what.

## Step 4A — Check training inputs and load the script

This prepares the variables used by the compatibility checks. It does not start training. If the cloned script already contains this definition, the cell leaves it unchanged and remains safe to rerun.


In [16]:
import os
import re
import subprocess
import sys
from pathlib import Path

root = Path(os.environ.get("HIGGS_REPO_ROOT", Path.cwd() / "higgs-event-lab")).resolve()
script = root / "scripts" / "train_full_colab.py"
training_sample_path = root / "data" / "processed" / "higgs_sample_1m.csv.gz"

if not script.is_file():
    raise FileNotFoundError(f"Training script not found: {script}. Run Step 1 first.")
if not training_sample_path.is_file():
    raise FileNotFoundError(f"Training sample not found: {training_sample_path}. Run Step 3B first.")

original_source = script.read_text(encoding="utf-8")
script_source = original_source
print("Training script:", script)
print("Training sample:", training_sample_path)

Training script: /kaggle/working/higgs-event-lab/scripts/train_full_colab.py
Training sample: /kaggle/working/higgs-event-lab/data/processed/higgs_sample_1m.csv.gz


## Step 4B — Fix `OUTPUTS_DIR` if it is missing

This addresses the previous `NameError: OUTPUTS_DIR is not defined` error. Run in order after Step 4A. This compatibility step keeps older repository copies runnable while preserving the script syntax before training begins.


In [17]:
header = script_source.split("def evaluate_run", 1)[0]
if not re.search(r"(?m)^OUTPUTS_DIR\s*=", header):
    root_line = re.search(r"(?m)^ROOT_DIR\s*=.*$", header)
    if not root_line:
        raise RuntimeError("Could not safely patch the training script: ROOT_DIR definition was not found.")
    insert_at = root_line.end()
    script_source = (
        script_source[:insert_at]
        + '\nOUTPUTS_DIR = ROOT_DIR / "outputs"\nOUTPUTS_DIR.mkdir(parents=True, exist_ok=True)'
        + script_source[insert_at:]
    )
else:
    header = script_source.split("def evaluate_run", 1)[0]
    if not re.search(r"(?m)^OUTPUTS_DIR\.mkdir\(", header):
        assign = re.search(r"(?m)^OUTPUTS_DIR\s*=.*$", header)
        if assign:
            script_source = (
                script_source[:assign.end()]
                + "\nOUTPUTS_DIR.mkdir(parents=True, exist_ok=True)"
                + script_source[assign.end():]
            )

print("OUTPUTS_DIR compatibility check complete.")

OUTPUTS_DIR compatibility check complete.


## Step 4C — Fix missing metric imports and save the patch

This addresses the previous `NameError: roc_auc_score is not defined` error. The training script is changed only if a fix is needed.

In [18]:
metrics_import = "from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score"
header = script_source.split("def evaluate_run", 1)[0]
if metrics_import not in header:
    pandas_import = re.search(r"(?m)^import pandas as pd\s*$", header)
    if pandas_import:
        script_source = script_source[:pandas_import.end()] + "\n" + metrics_import + script_source[pandas_import.end():]
    else:
        first_function = re.search(r"(?m)^def evaluate_run\b", script_source)
        if not first_function:
            raise RuntimeError("Could not safely patch metric imports in the training script.")
        script_source = script_source[:first_function.start()] + metrics_import + "\n\n" + script_source[first_function.start():]

 Syntax-check the source before writing it back to the repository.
compile(script_source, str(script), "exec")
if script_source != original_source:
    script.write_text(script_source, encoding="utf-8")
    print("Saved compatibility fixes to:", script)
else:
    print("The training script already contains the required definitions.")

(root / "outputs").mkdir(parents=True, exist_ok=True)

Saved compatibility fixes to: /kaggle/working/higgs-event-lab/scripts/train_full_colab.py


## Step 4D — Start model training

This is the long-running step. It runs the training script as a separate process. If it fails, read the traceback/log above; you do not need to redownload or re-extract the dataset for a training-only error.

In [19]:
env = os.environ.copy()
env["HIGGS_REPO_ROOT"] = str(root)
env["HIGGS_SAMPLE_PATH"] = str(training_sample_path)

print("Project root:", root)
print("Training sample:", training_sample_path)
print("Starting training...")
result = subprocess.run(
    [sys.executable, str(script), "--sample", str(training_sample_path)],
    cwd=root,
    env=env,
    check=False,
)
if result.returncode != 0:
    raise RuntimeError(
        f"Training failed with exit code {result.returncode}. See the log above. "
        "The data download and sample preparation do not need to be repeated."
    )
print("Training process finished successfully.")

Project root: /kaggle/working/higgs-event-lab
Training sample: /kaggle/working/higgs-event-lab/data/processed/higgs_sample_1m.csv.gz
Starting training...
Loaded 1,000,000 rows from higgs_sample_1m.csv.gz
Split sizes -> train 700,000, validation 150,000, test 150,000
Summary written to outputs/full_colab_results.json

Results (frozen test split):
  logistic_regression_tuned    ROC-AUC 0.6834  PR-AUC 0.6832  Brier 0.2233  accuracy 0.6406  seconds 2.6
  xgboost_tuned                ROC-AUC 0.8271  PR-AUC 0.8415  Brier 0.1693  accuracy 0.7451  seconds 35.1
  wide_mlp_tuned               ROC-AUC 0.8399  PR-AUC 0.8539  Brier 0.1630  accuracy 0.7565  seconds 395.1
Training process finished successfully.


## Step 4E — Confirm the results file exists

This is a quick checkpoint before displaying the metrics. This checkpoint prevents the results-display cells from hiding a failed or incomplete training process.


In [20]:
summary_path = root / "outputs" / "full_colab_results.json"
if not summary_path.is_file():
    raise FileNotFoundError(f"Training finished but summary was not created: {summary_path}")
print("Training summary found:", summary_path)

Training summary found: /kaggle/working/higgs-event-lab/outputs/full_colab_results.json


## Step 5A — Load the saved results

Run after training completes. These values are the report-ready test metrics and should be copied only from the saved JSON summary.


In [21]:
import json
from pathlib import Path

summary_path = root / "outputs" / "full_colab_results.json"
if not summary_path.exists():
    raise FileNotFoundError("No training summary found. Run Step 4D first.")
summary = json.loads(summary_path.read_text(encoding="utf-8"))
print("Sample rows:", summary["rows"])
print("Split:", summary["split"])

Sample rows: 1000000
Split: {'seed': 42, 'validation_fraction': 0.15, 'test_fraction': 0.15}


## Step 5B — Display model metrics The listed files are the evidence package for the report, dashboard and later reproducibility review. The table is the report-ready comparison of held-out performance and runtime for the three Protocol B models.


In [22]:
print(f"{'Model':<28} {'ROC-AUC':>8} {'PR-AUC':>8} {'Brier':>8} {'Accuracy':>9} {'Seconds':>9}")
print("-" * 80)
for record in summary["runs"]:
    print(
        f"{record['model']:<28} "
        f"{record['test_roc_auc']:>8.4f} "
        f"{record['test_pr_auc']:>8.4f} "
        f"{record['brier_score']:>8.4f} "
        f"{record['test_accuracy']:>9.4f} "
        f"{record['training_seconds']:>9.1f}"
    )

Model                         ROC-AUC   PR-AUC    Brier  Accuracy   Seconds
--------------------------------------------------------------------------------
logistic_regression_tuned      0.6834   0.6832   0.2233    0.6406       2.6
xgboost_tuned                  0.8271   0.8415   0.1693    0.7451      35.1
wide_mlp_tuned                 0.8399   0.8539   0.1630    0.7565     395.1


## Step 5C — List saved artifacts This inventory lets the team attach the generated JSON, prediction tables and model files as reproducibility evidence. Confirm that these paths exist before downloading the results from the temporary cloud runtime.


In [23]:
print("Artifacts:")
for artifact in summary.get("artifacts", []):
    print(" ", artifact)

Artifacts:
  outputs/logistic_regression_tuned_test_predictions.csv
  outputs/xgboost_tuned_test_predictions.csv
  outputs/wide_mlp_tuned_test_predictions.csv
  outputs/model_weights/feature_scaler.joblib
  outputs/model_weights/logistic_regression_tuned.joblib
  outputs/model_weights/wide_mlp_tuned.pt
  outputs/model_weights/xgboost_tuned.joblib
